<a href="https://colab.research.google.com/github/Shushhere/Compiler_Design_2472182/blob/main/compiler_design_2472182.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1.lexical Analysis

In [ ]:
# Lexical Analysis Implementation

import re
import keyword

def scan_source_code(source_text):
    # Extracting potential lexemes using a distinct regex
    lexemes = re.findall(
        r'\b\d+\b|[a-zA-Z_]\w*|==|!=|<=|>=|[+\-*/%=<>]|[(){};,]',
        source_text
    )

    print(f"{'LEXEME':<16}{'TOKEN CLASS'}")
    print("-" * 32)

    for lexeme in lexemes:
        if keyword.iskeyword(lexeme):
            classification = "Keyword"
        elif lexeme.isidentifier():
            classification = "Identifier"
        elif lexeme.isdigit():
            classification = "Numeric Constant"
        elif lexeme in {"+", "-", "*", "/", "%", "=", "<", ">", "==", "!=", "<=", ">="}:
            classification = "Operator"
        else:
            classification = "Punctuation/Symbol"

        print(f"{lexeme:<16}{classification}")

source_program = """
int x = 10;
int y = 20;
if (x < y) {
    x = x + y;
}
"""

scan_source_code(source_program)

LEXEME          TOKEN CLASS
--------------------------------
int             Identifier
x               Identifier
=               Operator
10              Numeric Constant
;               Punctuation/Symbol
int             Identifier
y               Identifier
=               Operator
20              Numeric Constant
;               Punctuation/Symbol
if              Keyword
(               Punctuation/Symbol
x               Identifier
<               Operator
y               Identifier
)               Punctuation/Symbol
{               Punctuation/Symbol
x               Identifier
=               Operator
x               Identifier
+               Operator
y               Identifier
;               Punctuation/Symbol
}               Punctuation/Symbol


2.First() and Follow()

In [ ]:
# First and Follow Sets Calculation

syntax_rules = {
    'E': ['TR'],
    'R': ['+TR', '#'],
    'T': ['FY'],
    'Y': ['*FY', '#'],
    'F': ['(E)', 'i']
}

first_map = {non_term: set() for non_term in syntax_rules}
follow_map = {non_term: set() for non_term in syntax_rules}

def compute_first(sym):
    if sym not in syntax_rules:
        return {sym}

    if first_map[sym]:
        return first_map[sym]

    for option in syntax_rules[sym]:
        if option == '#':
            first_map[sym].add('#')
            continue

        for unit in option:
            subset = compute_first(unit)
            first_map[sym].update(subset - {'#'})
            if '#' not in subset:
                break
        else:
            first_map[sym].add('#')

    return first_map[sym]

def compute_follow():
    follow_map['E'].add('$')
    is_updated = True

    while is_updated:
        is_updated = False
        for head, bodies in syntax_rules.items():
            for body in bodies:
                for pos, element in enumerate(body):
                    if element in syntax_rules:
                        prev_count = len(follow_map[element])
                        trailing = body[pos + 1:]

                        if trailing:
                            next_firsts = set()
                            can_be_null = True

                            for sub_sym in trailing:
                                sub_first = compute_first(sub_sym)
                                next_firsts.update(sub_first - {'#'})
                                if '#' not in sub_first:
                                    can_be_null = False
                                    break

                            follow_map[element].update(next_firsts)
                            if can_be_null:
                                follow_map[element].update(follow_map[head])
                        else:
                            follow_map[element].update(follow_map[head])

                        if len(follow_map[element]) > prev_count:
                            is_updated = True

for nt in syntax_rules:
    compute_first(nt)

compute_follow()

print("FIRST SETS:")
for key, values in first_map.items():
    print(f"FIRST( {key} ) = {values}")

print("\nFOLLOW SETS:")
for key, values in follow_map.items():
    print(f"FOLLOW( {key} ) = {values}")

FIRST SETS:
FIRST( E ) = {'(', 'i'}
FIRST( R ) = {'+', '#'}
FIRST( T ) = {'(', 'i'}
FIRST( Y ) = {'*', '#'}
FIRST( F ) = {'(', 'i'}

FOLLOW SETS:
FOLLOW( E ) = {'$', ')'}
FOLLOW( R ) = {'$', ')'}
FOLLOW( T ) = {'+', '$', ')'}
FOLLOW( Y ) = {'+', '$', ')'}
FOLLOW( F ) = {'+', '*', '$', ')'}


LL(1) parser

In [ ]:
# LL(1) Parser Implementation

# Configuration of Parsing Table
ll1_dispatch_table = {
    'E': {'i': 'TQ', '(': 'TQ'},
    'Q': {'+': '+TQ', ')': '#', '$': '#'},
    'T': {'i': 'FR', '(': 'FR'},
    'R': {'+': '#', '*': '*FR', ')': '#', '$': '#'},
    'F': {'i': 'i', '(': '(E)'}
}

valid_ends = {'i', '+', '*', '(', ')', '$'}

def execute_ll1_parse(expr_str):
    # Initialize symbol stack and target array
    token_stack = ['$', 'E']
    buffered_expr = expr_str + '$'
    char_ptr = 0

    print(f"\n{'STACK':<16}{'INPUT STREAM':<16}{'PARSER ACTION'}")
    print("-" * 50)

    while len(token_stack) > 0:
        top_symbol = token_stack[-1]
        current_char = buffered_expr[char_ptr]

        stack_view = ''.join(token_stack)
        input_view = buffered_expr[char_ptr:]
        print(f"{stack_view:<16}{input_view:<16}", end='')

        if top_symbol == current_char == '$':
            print("SUCCESS: STRING ACCEPTED")
            return True

        if top_symbol == current_char:
            token_stack.pop()
            char_ptr += 1
            print(f"Matched token: {current_char}")

        elif top_symbol in ll1_dispatch_table:
            production_rule = ll1_dispatch_table[top_symbol].get(current_char)

            if production_rule is None:
                print("REJECTED: No production rule available")
                return False

            token_stack.pop()

            if production_rule != '#':
                for char in reversed(production_rule):
                    token_stack.append(char)

            print(f"{top_symbol} -> {production_rule}")
        else:
            print("REJECTED: Invalid symbol transition")
            return False

    print("REJECTED: Extra tokens remaining")
    return False

user_input = input("Enter expression to parse (e.g., i+i*i): ")
execute_ll1_parse(user_input)

Enter expression to parse (e.g., i+i*i): i+i+i

STACK           INPUT STREAM    PARSER ACTION
--------------------------------------------------
$E              i+i+i$          E -> TQ
$QT             i+i+i$          T -> FR
$QRF            i+i+i$          F -> i
$QRi            i+i+i$          Matched token: i
$QR             +i+i$           R -> #
$Q              +i+i$           Q -> +TQ
$QT+            +i+i$           Matched token: +
$QT             i+i$            T -> FR
$QRF            i+i$            F -> i
$QRi            i+i$            Matched token: i
$QR             +i$             R -> #
$Q              +i$             Q -> +TQ
$QT+            +i$             Matched token: +
$QT             i$              T -> FR
$QRF            i$              F -> i
$QRi            i$              Matched token: i
$QR             $               R -> #
$Q              $               Q -> #
$               $               SUCCESS: STRING ACCEPTED


True

Recursive Descent parser

In [ ]:
# Recursive Descent Parser Implementation

input_stream = input("Enter the input string (use i for identifier): ")
cursor_position = 0

def advance_cursor(expected_symbol):
    global cursor_position
    if cursor_position < len(input_stream) and input_stream[cursor_position] == expected_symbol:
        cursor_position += 1
        return True
    return False

# E -> TQ
def parse_E():
    if parse_T():
        if parse_Q():
            return True
    return False

# Q -> +TQ | epsilon
def parse_Q():
    if advance_cursor('+'):
        if parse_T():
            return parse_Q()
        return False
    return True

# T -> FR
def parse_T():
    if parse_F():
        if parse_R():
            return True
    return False

# R -> *FR | epsilon
def parse_R():
    if advance_cursor('*'):
        if parse_F():
            return parse_R()
        return False
    return True

# F -> (E) | i
def parse_F():
    if advance_cursor('i'):
        return True
    elif advance_cursor('('):
        if parse_E():
            if advance_cursor(')'):
                return True
    return False

# Execute parser evaluation
if parse_E() and cursor_position == len(input_stream):
    print("String Accepted")
else:
    print("String Rejected")

Enter the input string (use i for identifier): i+i*i
String Accepted


### Parser Robustness & Error Handling Tests
Below, we define a list of invalid inputs and test both the LL(1) and Recursive Descent parsers to ensure they handle syntax errors correctly without raising unhandled runtime exceptions.

In [ ]:
# List of invalid input scenarios to test both parsers

invalid_test_cases = [
    "i+",         # Trailing operator
    "(i+i",       # Unmatched left parenthesis
    "i+i)",       # Unmatched right parenthesis
    "i*+i",       # Consecutive operators
    "+i*i",       # Leading operator
    "()",         # Empty parentheses
    "i a"         # Unrecognized spaces/symbols
]

print("=== TESTING LL(1) PARSER WITH INVALID INPUTS ===")
for case in invalid_test_cases:
    print(f"\nTesting input: '{case}'")
    # Diverting stdout to run quietly and print only the final assertion status
    result = execute_ll1_parse(case)
    print(f"Result: {'REJECTED (Correct)' if not result else 'ACCEPTED (Incorrect)'}")

print("\n=== TESTING RECURSIVE DESCENT PARSER WITH INVALID INPUTS ===")
for case in invalid_test_cases:
    # Setup global parser state for each iteration
    input_stream = case
    cursor_position = 0

    # Run recursive parser
    success = parse_E() and cursor_position == len(input_stream)
    print(f"Testing input: '{case}' -> Result: {'REJECTED (Correct)' if not success else 'ACCEPTED (Incorrect)'}")

=== TESTING LL(1) PARSER WITH INVALID INPUTS ===

Testing input: 'i+'

STACK           INPUT STREAM    PARSER ACTION
--------------------------------------------------
$E              i+$             E -> TQ
$QT             i+$             T -> FR
$QRF            i+$             F -> i
$QRi            i+$             Matched token: i
$QR             +$              R -> #
$Q              +$              Q -> +TQ
$QT+            +$              Matched token: +
$QT             $               REJECTED: No production rule available
Result: REJECTED (Correct)

Testing input: '(i+i'

STACK           INPUT STREAM    PARSER ACTION
--------------------------------------------------
$E              (i+i$           E -> TQ
$QT             (i+i$           T -> FR
$QRF            (i+i$           F -> (E)
$QR)E(          (i+i$           Matched token: (
$QR)E           i+i$            E -> TQ
$QR)QT          i+i$            T -> FR
$QR)QRF         i+i$            F -> i
$QR)QRi         i+i$       

LR(0) PARSER

In [ ]:
# Program to Check LR(0) Grammar

cfg_grammar = {
    'S': ['CC'],
    'C': ['cC', 'd']
}

augmented_rules = [("S'", "S")]
for non_term in cfg_grammar:
    for production in cfg_grammar[non_term]:
        augmented_rules.append((non_term, production))

nt_set = set(cfg_grammar.keys()) | {"S'"}
t_set = set()

for lhs, rhs in augmented_rules:
    for character in rhs:
        if character not in nt_set:
            t_set.add(character)
t_set.add('$')

def find_closure(item_set):
    active_items = set(item_set)
    has_changed = True
    while has_changed:
        has_changed = False
        for rule_idx, dot_pos in list(active_items):
            lhs, rhs = augmented_rules[rule_idx]
            if dot_pos < len(rhs):
                next_symbol = rhs[dot_pos]
                if next_symbol in nt_set:
                    for idx, (l, r) in enumerate(augmented_rules):
                        if l == next_symbol and (idx, 0) not in active_items:
                            active_items.add((idx, 0))
                            has_changed = True
    return frozenset(active_items)

def compute_goto(item_set, symbol):
    shifted_items = set()
    for rule_idx, dot_pos in item_set:
        lhs, rhs = augmented_rules[rule_idx]
        if dot_pos < len(rhs) and rhs[dot_pos] == symbol:
            shifted_items.add((rule_idx, dot_pos + 1))
    if shifted_items:
        return find_closure(shifted_items)
    return frozenset()

lr0_states = [find_closure({(0, 0)})]
state_transitions = {}

index = 0
while index < len(lr0_states):
    for sym in sorted(t_set | nt_set - {'$', "S'"}):
        target_state = compute_goto(lr0_states[index], sym)
        if target_state:
            if target_state not in lr0_states:
                lr0_states.append(target_state)
            state_transitions[(index, sym)] = lr0_states.index(target_state)
    index += 1

print("LR(0) ITEM SETS")
for state_id, state in enumerate(lr0_states):
    print(f"\nI{state_id}")
    for rule_idx, dot_pos in sorted(state):
        lhs, rhs = augmented_rules[rule_idx]
        item_representation = rhs[:dot_pos] + "." + rhs[dot_pos:]
        print(f"{lhs} -> {item_representation}")

action_table = {}
goto_table = {}

for state_id in range(len(lr0_states)):
    for terminal in t_set:
        action_table[(state_id, terminal)] = ''
    for non_term in cfg_grammar:
        goto_table[(state_id, non_term)] = ''

has_conflict = False

def update_action_cell(cell_key, cell_val):
    global has_conflict
    if action_table[cell_key] != '' and action_table[cell_key] != cell_val:
        has_conflict = True
    else:
        action_table[cell_key] = cell_val

for state_id, state in enumerate(lr0_states):
    for terminal in t_set - {'$'}:
        if (state_id, terminal) in state_transitions:
            update_action_cell((state_id, terminal), f"S{state_transitions[(state_id, terminal)]}")

    for non_term in cfg_grammar:
        if (state_id, non_term) in state_transitions:
            goto_table[(state_id, non_term)] = state_transitions[(state_id, non_term)]

    for rule_idx, dot_pos in state:
        lhs, rhs = augmented_rules[rule_idx]
        if dot_pos == len(rhs):
            if rule_idx == 0:
                update_action_cell((state_id, '$'), 'ACCEPT')
            else:
                for terminal in t_set:
                    update_action_cell((state_id, terminal), f"R{rule_idx}")

print("\nACTION TABLE")
for state_id in range(len(lr0_states)):
    print(f"State {state_id} : ", end="")
    for terminal in sorted(t_set):
        print(f"{terminal} = {action_table[(state_id, terminal)]:<8}", end="")
    print()

print("\nGOTO TABLE")
for state_id in range(len(lr0_states)):
    print(f"State {state_id} : ", end="")
    for non_term in cfg_grammar:
        print(f"{non_term} = {goto_table[(state_id, non_term)]:<8}", end="")
    print()

if has_conflict:
    print("\nGrammar is NOT LR(0)")
else:
    print("\nGrammar IS LR(0)")

LR(0) ITEM SETS

I0
S' -> .S
S -> .CC
C -> .cC
C -> .d

I1
S -> C.C
C -> .cC
C -> .d

I2
S' -> S.

I3
C -> .cC
C -> c.C
C -> .d

I4
C -> d.

I5
S -> CC.

I6
C -> cC.

ACTION TABLE
State 0 : $ =         c = S3      d = S4      
State 1 : $ =         c = S3      d = S4      
State 2 : $ = ACCEPT  c =         d =         
State 3 : $ =         c = S3      d = S4      
State 4 : $ = R3      c = R3      d = R3      
State 5 : $ = R1      c = R1      d = R1      
State 6 : $ = R2      c = R2      d = R2      

GOTO TABLE
State 0 : S = 2       C = 1       
State 1 : S =         C = 5       
State 2 : S =         C =         
State 3 : S =         C = 6       
State 4 : S =         C =         
State 5 : S =         C =         
State 6 : S =         C =         

Grammar IS LR(0)


SLR(1) PARSER

In [ ]:
# Program to Check SLR(1) Grammar

target_grammar = {
    'S': ['CC'],
    'C': ['cC', 'd']
}

rule_mappings = [("S'", "S")]
for non_terminal in target_grammar:
    for rhs in target_grammar[non_terminal]:
        rule_mappings.append((non_terminal, rhs))

nt_keys = set(target_grammar.keys()) | {"S'"}
t_keys = set()
for left, right in rule_mappings:
    for char in right:
        if char not in nt_keys:
            t_keys.add(char)
t_keys.add('$')

def get_first_set(symbol):
    if symbol not in target_grammar:
        return {symbol}

    firsts = set()
    for prod in target_grammar[symbol]:
        if prod == '#':
            firsts.add('#')
            continue

        is_nullable = True
        for char in prod:
            child_firsts = get_first_set(char)
            firsts.update(child_firsts - {'#'})
            if '#' not in child_firsts:
                is_nullable = False
                break
        if is_nullable:
            firsts.add('#')
    return firsts

slr_follows = {nt: set() for nt in target_grammar}
slr_follows['S'].add('$')

is_changing = True
while is_changing:
    is_changing = False
    for left, right in rule_mappings[1:]:
        for idx, char in enumerate(right):
            if char in target_grammar:
                old_len = len(slr_follows[char])
                remainder = right[idx + 1:]

                if remainder:
                    nullable = True
                    for item in remainder:
                        temp_first = get_first_set(item)
                        slr_follows[char].update(temp_first - {'#'})
                        if '#' not in temp_first:
                            nullable = False
                            break
                    if nullable:
                        slr_follows[char].update(slr_follows[left])
                else:
                    slr_follows[char].update(slr_follows[left])

                if len(slr_follows[char]) > old_len:
                    is_changing = True

def slr_closure(items):
    items_set = set(items)
    updating = True
    while updating:
        updating = False
        for p_idx, dot in list(items_set):
            left, right = rule_mappings[p_idx]
            if dot < len(right):
                next_sym = right[dot]
                if next_sym in nt_keys:
                    for idx, (l_side, r_side) in enumerate(rule_mappings):
                        if l_side == next_sym and (idx, 0) not in items_set:
                            items_set.add((idx, 0))
                            updating = True
    return frozenset(items_set)

def slr_goto(items, symbol):
    moved_items = set()
    for p_idx, dot in items:
        left, right = rule_mappings[p_idx]
        if dot < len(right) and right[dot] == symbol:
            moved_items.add((p_idx, dot + 1))
    return slr_closure(moved_items) if moved_items else frozenset()

slr_states = [slr_closure({(0, 0)})]
state_links = {}

state_idx = 0
while state_idx < len(slr_states):
    for sym in sorted((t_keys | nt_keys) - {'$', "S'"}):
        next_state = slr_goto(slr_states[state_idx], sym)
        if next_state:
            if next_state not in slr_states:
                slr_states.append(next_state)
            state_links[(state_idx, sym)] = slr_states.index(next_state)
    state_idx += 1

actions = {}
gotos = {}

for i in range(len(slr_states)):
    for t in t_keys:
        actions[(i, t)] = ''
    for nt in target_grammar:
        gotos[(i, nt)] = ''

has_slr_conflict = False

def add_slr_action(state, term, val):
    global has_slr_conflict
    if actions[(state, term)] != '' and actions[(state, term)] != val:
        has_slr_conflict = True
    else:
        actions[(state, term)] = val

for i, state in enumerate(slr_states):
    for t in t_keys - {'$'}:
        if (i, t) in state_links:
            add_slr_action(i, t, f"S{state_links[(i, t)]}")

    for nt in target_grammar:
        if (i, nt) in state_links:
            gotos[(i, nt)] = state_links[(i, nt)]

    for p_idx, dot in state:
        left, right = rule_mappings[p_idx]
        if dot == len(right):
            if p_idx == 0:
                add_slr_action(i, '$', 'ACCEPT')
            else:
                for t in slr_follows[left]:
                    add_slr_action(i, t, f"R{p_idx}")

print("FOLLOW SETS")
for nt in target_grammar:
    print(f"FOLLOW({nt}) = {slr_follows[nt]}")

print("\nLR(0) ITEM SETS")
for i, state in enumerate(slr_states):
    print(f"\nI{i}")
    for p_idx, dot in sorted(state):
        left, right = rule_mappings[p_idx]
        print(f"{left} -> {right[:dot]}.{right[dot:]}")

print("\nACTION TABLE")
for i in range(len(slr_states)):
    print(f"State {i} : ", end="")
    for t in sorted(t_keys):
        print(f"{t} = {actions[(i, t)]:<6} ", end="")
    print()

print("\nGOTO TABLE")
for i in range(len(slr_states)):
    print(f"State {i} : ", end="")
    for nt in target_grammar:
        print(f"{nt} = {gotos[(i, nt)]:<6} ", end="")
    print()

if has_slr_conflict:
    print("\nGrammar is NOT SLR(1)")
else:
    print("\nGrammar IS SLR(1)")

FOLLOW SETS
FOLLOW(S) = {'$'}
FOLLOW(C) = {'$', 'c', 'd'}

LR(0) ITEM SETS

I0
S' -> .S
S -> .CC
C -> .cC
C -> .d

I1
S -> C.C
C -> .cC
C -> .d

I2
S' -> S.

I3
C -> .cC
C -> c.C
C -> .d

I4
C -> d.

I5
S -> CC.

I6
C -> cC.

ACTION TABLE
State 0 : $ =        c = S3     d = S4     
State 1 : $ =        c = S3     d = S4     
State 2 : $ = ACCEPT c =        d =        
State 3 : $ =        c = S3     d = S4     
State 4 : $ = R3     c = R3     d = R3     
State 5 : $ = R1     c =        d =        
State 6 : $ = R2     c = R2     d = R2     

GOTO TABLE
State 0 : S = 2      C = 1      
State 1 : S =        C = 5      
State 2 : S =        C =        
State 3 : S =        C = 6      
State 4 : S =        C =        
State 5 : S =        C =        
State 6 : S =        C =        

Grammar IS SLR(1)


CLR PARSER

In [ ]:
# Program to Check CLR(1) Grammar

clr_grammar = {
    'S': ['CC'],
    'C': ['cC', 'd']
}

clr_augmented_rules = [("S'", "S")]
for non_terminal in clr_grammar:
    for rhs in clr_grammar[non_terminal]:
        clr_augmented_rules.append((non_terminal, rhs))

clr_nt = set(clr_grammar.keys()) | {"S'"}
clr_t = set()

for left, right in clr_augmented_rules:
    for char in right:
        if char not in clr_nt:
            clr_t.add(char)
clr_t.add('$')

def compute_clr_first(sequence):
    firsts = set()
    if not sequence:
        return {'#'}

    for symbol in sequence:
        if symbol not in clr_nt:
            firsts.add(symbol)
            return firsts

        is_nullable = False
        for rule in clr_grammar[symbol]:
            if rule == '#':
                is_nullable = True
            else:
                child_firsts = compute_clr_first(rule)
                firsts.update(child_firsts - {'#'})
                if '#' in child_firsts:
                    is_nullable = True

        if not is_nullable:
            return firsts

    firsts.add('#')
    return firsts

def clr_closure(items):
    items_set = set(items)
    is_updating = True

    while is_updating:
        is_updating = False
        for p_idx, dot, lookahead in list(items_set):
            left, right = clr_augmented_rules[p_idx]
            if dot < len(right):
                next_symbol = right[dot]
                if next_symbol in clr_nt:
                    beta = right[dot + 1:] + lookahead
                    sub_lookaheads = compute_clr_first(beta) - {'#'}
                    for idx, (l_side, r_side) in enumerate(clr_augmented_rules):
                        if l_side == next_symbol:
                            for term in sub_lookaheads:
                                new_item = (idx, 0, term)
                                if new_item not in items_set:
                                    items_set.add(new_item)
                                    is_updating = True
    return frozenset(items_set)

def clr_goto(items, symbol):
    moved_items = set()
    for p_idx, dot, lookahead in items:
        left, right = clr_augmented_rules[p_idx]
        if dot < len(right) and right[dot] == symbol:
            moved_items.add((p_idx, dot + 1, lookahead))
    return clr_closure(moved_items) if moved_items else frozenset()

clr_states = [clr_closure({(0, 0, '$')})]
clr_state_links = {}

state_idx = 0
while state_idx < len(clr_states):
    possible_symbols = sorted((clr_t | clr_nt) - {'$'}
                              if '$' not in (clr_t | clr_nt)
                              else sorted((clr_t | clr_nt) - {'$'}))
    for sym in possible_symbols:
        next_state = clr_goto(clr_states[state_idx], sym)
        if next_state:
            if next_state not in clr_states:
                clr_states.append(next_state)
            clr_state_links[(state_idx, sym)] = clr_states.index(next_state)
    state_idx += 1

clr_actions = {}
clr_gotos = {}

for i in range(len(clr_states)):
    for t in clr_t:
        clr_actions[(i, t)] = ''
    for nt in clr_grammar:
        clr_gotos[(i, nt)] = ''

has_clr_conflict = False

def add_clr_action(state, term, val):
    global has_clr_conflict
    if clr_actions[(state, term)] != '' and clr_actions[(state, term)] != val:
        has_clr_conflict = True
    else:
        clr_actions[(state, term)] = val

for i, state in enumerate(clr_states):
    for t in clr_t - {'$'}:
        if (i, t) in clr_state_links:
            add_clr_action(i, t, f"S{clr_state_links[(i, t)]}")

    for nt in clr_grammar:
        if (i, nt) in clr_state_links:
            clr_gotos[(i, nt)] = clr_state_links[(i, nt)]

    for p_idx, dot, lookahead in state:
        left, right = clr_augmented_rules[p_idx]
        if dot == len(right):
            if p_idx == 0:
                add_clr_action(i, '$', 'ACCEPT')
            else:
                add_clr_action(i, lookahead, f"R{p_idx}")

print("CLR(1) ITEM SETS")
for i, state in enumerate(clr_states):
    print(f"\nI{i}")
    for p_idx, dot, lookahead in sorted(state):
        left, right = clr_augmented_rules[p_idx]
        print(f"{left} -> {right[:dot]}.{right[dot:]} , {lookahead}")

print("\nACTION TABLE")
for i in range(len(clr_states)):
    print(f"State {i} : ", end="")
    for t in sorted(clr_t):
        print(f"{t} = {clr_actions[(i, t)]:<6} ", end="")
    print()

print("\nGOTO TABLE")
for i in range(len(clr_states)):
    print(f"State {i} : ", end="")
    for nt in clr_grammar:
        print(f"{nt} = {clr_gotos[(i, nt)]:<6} ", end="")
    print()

if has_clr_conflict:
    print("\nGrammar is NOT CLR(1)")
else:
    print("\nGrammar IS CLR(1)")

CLR(1) ITEM SETS

I0
S' -> .S , $
S -> .CC , $
C -> .cC , c
C -> .cC , d
C -> .d , c
C -> .d , d

I1
S -> C.C , $
C -> .cC , $
C -> .d , $

I2
S' -> S. , $

I3
C -> .cC , c
C -> .cC , d
C -> c.C , c
C -> c.C , d
C -> .d , c
C -> .d , d

I4
C -> d. , c
C -> d. , d

I5
S -> CC. , $

I6
C -> .cC , $
C -> c.C , $
C -> .d , $

I7
C -> d. , $

I8
C -> cC. , c
C -> cC. , d

I9
C -> cC. , $

ACTION TABLE
State 0 : $ =        c = S3     d = S4     
State 1 : $ =        c = S6     d = S7     
State 2 : $ = ACCEPT c =        d =        
State 3 : $ =        c = S3     d = S4     
State 4 : $ =        c = R3     d = R3     
State 5 : $ = R1     c =        d =        
State 6 : $ =        c = S6     d = S7     
State 7 : $ = R3     c =        d =        
State 8 : $ =        c = R2     d = R2     
State 9 : $ = R2     c =        d =        

GOTO TABLE
State 0 : S = 2      C = 1      
State 1 : S =        C = 5      
State 2 : S =        C =        
State 3 : S =        C = 8      
State 4 : S =   

LALR PARSER

In [ ]:
# Program to Check LALR(1) Grammar

lalr_grammar = {
    'S': ['CC'],
    'C': ['cC', 'd']
}

lalr_augmented_rules = [("S'", "S")]
for non_terminal in lalr_grammar:
    for rhs in lalr_grammar[non_terminal]:
        lalr_augmented_rules.append((non_terminal, rhs))

lalr_nt = set(lalr_grammar.keys()) | {"S'"}
lalr_t = set()

for left, right in lalr_augmented_rules:
    for char in right:
        if char not in lalr_nt:
            lalr_t.add(char)
lalr_t.add('$')

def compute_lalr_first(sequence):
    firsts = set()
    if not sequence:
        return {'#'}

    for symbol in sequence:
        if symbol not in lalr_nt:
            firsts.add(symbol)
            return firsts

        is_nullable = False
        for rule in lalr_grammar[symbol]:
            if rule == '#':
                is_nullable = True
            else:
                child_firsts = compute_lalr_first(rule)
                firsts.update(child_firsts - {'#'})
                if '#' in child_firsts:
                    is_nullable = True

        if not is_nullable:
            return firsts

    firsts.add('#')
    return firsts

def lalr_closure(items):
    items_set = set(items)
    is_updating = True

    while is_updating:
        is_updating = False
        for p_idx, dot, lookahead in list(items_set):
            left, right = lalr_augmented_rules[p_idx]
            if dot < len(right):
                next_symbol = right[dot]
                if next_symbol in lalr_nt:
                    beta = right[dot + 1:] + lookahead
                    sub_lookaheads = compute_lalr_first(beta) - {'#'}
                    for idx, (l_side, r_side) in enumerate(lalr_augmented_rules):
                        if l_side == next_symbol:
                            for term in sub_lookaheads:
                                new_item = (idx, 0, term)
                                if new_item not in items_set:
                                    items_set.add(new_item)
                                    is_updating = True
    return frozenset(items_set)

def lalr_goto(items, symbol):
    moved_items = set()
    for p_idx, dot, lookahead in items:
        left, right = lalr_augmented_rules[p_idx]
        if dot < len(right) and right[dot] == symbol:
            moved_items.add((p_idx, dot + 1, lookahead))
    return lalr_closure(moved_items) if moved_items else frozenset()

clr_states = [lalr_closure({(0, 0, '$')})]
clr_state_links = {}

state_idx = 0
while state_idx < len(clr_states):
    possible_symbols = sorted((lalr_t | lalr_nt) - {'$'})
    for sym in possible_symbols:
        next_state = lalr_goto(clr_states[state_idx], sym)
        if next_state:
            if next_state not in clr_states:
                clr_states.append(next_state)
            clr_state_links[(state_idx, sym)] = clr_states.index(next_state)
    state_idx += 1

def get_lr0_core(item_set):
    return frozenset((p, d) for p, d, la in item_set)

lalr_states = []
core_to_index = {}
clr_to_lalr_map = {}

for idx, clr_state in enumerate(clr_states):
    core_items = get_lr0_core(clr_state)
    if core_items not in core_to_index:
        core_to_index[core_items] = len(lalr_states)
        lalr_states.append(set())

    lalr_target_idx = core_to_index[core_items]
    clr_to_lalr_map[idx] = lalr_target_idx
    lalr_states[lalr_target_idx].update(clr_state)

lalr_states = [frozenset(state) for state in lalr_states]

lalr_state_links = {}
for (from_state, sym), to_state in clr_state_links.items():
    new_from = clr_to_lalr_map[from_state]
    new_to = clr_to_lalr_map[to_state]
    if (new_from, sym) in lalr_state_links and lalr_state_links[(new_from, sym)] != new_to:
        raise ValueError("Invalid merged state transition conflict identified!")
    lalr_state_links[(new_from, sym)] = new_to

lalr_actions = {}
lalr_gotos = {}

for i in range(len(lalr_states)):
    for t in lalr_t:
        lalr_actions[(i, t)] = ''
    for nt in lalr_grammar:
        lalr_gotos[(i, nt)] = ''

has_lalr_conflict = False

def add_lalr_action(state, term, val):
    global has_lalr_conflict
    if lalr_actions[(state, term)] != '' and lalr_actions[(state, term)] != val:
        has_lalr_conflict = True
    else:
        lalr_actions[(state, term)] = val

for i, state in enumerate(lalr_states):
    for t in lalr_t - {'$'}:
        if (i, t) in lalr_state_links:
            add_lalr_action(i, t, f"S{lalr_state_links[(i, t)]}")

    for nt in lalr_grammar:
        if (i, nt) in lalr_state_links:
            lalr_gotos[(i, nt)] = lalr_state_links[(i, nt)]

    for p_idx, dot, lookahead in state:
        left, right = lalr_augmented_rules[p_idx]
        if dot == len(right):
            if p_idx == 0:
                add_lalr_action(i, '$', 'ACCEPT')
            else:
                add_lalr_action(i, lookahead, f"R{p_idx}")

print("LALR(1) ITEM SETS")
for i, state in enumerate(lalr_states):
    print(f"\nI{i}")
    for p_idx, dot, lookahead in sorted(state):
        left, right = lalr_augmented_rules[p_idx]
        print(f"{left} -> {right[:dot]}.{right[dot:]} , {lookahead}")

print("\nACTION TABLE")
for i in range(len(lalr_states)):
    print(f"State {i} : ", end="")
    for t in sorted(lalr_t):
        print(f"{t} = {lalr_actions[(i, t)]:<6} ", end="")
    print()

print("\nGOTO TABLE")
for i in range(len(lalr_states)):
    print(f"State {i} : ", end="")
    for nt in lalr_grammar:
        print(f"{nt} = {lalr_gotos[(i, nt)]:<6} ", end="")
    print()

if has_lalr_conflict:
    print("\nGrammar is NOT LALR(1)")
else:
    print("\nGrammar IS LALR(1)")

LALR(1) ITEM SETS

I0
S' -> .S , $
S -> .CC , $
C -> .cC , c
C -> .cC , d
C -> .d , c
C -> .d , d

I1
S -> C.C , $
C -> .cC , $
C -> .d , $

I2
S' -> S. , $

I3
C -> .cC , $
C -> .cC , c
C -> .cC , d
C -> c.C , $
C -> c.C , c
C -> c.C , d
C -> .d , $
C -> .d , c
C -> .d , d

I4
C -> d. , $
C -> d. , c
C -> d. , d

I5
S -> CC. , $

I6
C -> cC. , $
C -> cC. , c
C -> cC. , d

ACTION TABLE
State 0 : $ =        c = S3     d = S4     
State 1 : $ =        c = S3     d = S4     
State 2 : $ = ACCEPT c =        d =        
State 3 : $ =        c = S3     d = S4     
State 4 : $ = R3     c = R3     d = R3     
State 5 : $ = R1     c =        d =        
State 6 : $ = R2     c = R2     d = R2     

GOTO TABLE
State 0 : S = 2      C = 1      
State 1 : S =        C = 5      
State 2 : S =        C =        
State 3 : S =        C = 6      
State 4 : S =        C =        
State 5 : S =        C =        
State 6 : S =        C =        

Grammar IS LALR(1)
